In [1]:
!python --version

Python 3.11.14


In [2]:
import os, sys
import pandas as pd

from pathlib import Path

ROOT = Path().resolve().parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.calc_degs_lib import CALC_DEGS
from libs.tcga_gdc_lib import *
from libs.Basic import *


ROOT: /home/flavio/uv/perturb_agent
SRC added: /home/flavio/uv/perturb_agent/src


### Defaults

In [3]:
ROOT = Path().resolve().parent
root0 = ROOT / "data"

gdc = GDC(root0=root0)

os.listdir(root0)[:10]


['cancer', 'reactome', 'vector_store', 'TCGA', 'gdc_programs.txt']

### Get all programs

In [4]:
force=False
verbose=True

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)


File read at '/home/flavio/uv/perturb_agent/data/gdc_programs.txt'


In [5]:
np.array(prog_list)

array(['TCGA', 'MATCH', 'TARGET', 'CGCI', 'CMI', 'APOLLO', 'BEATAML1.0',
       'CPTAC', 'MP2PRT', 'ALCHEMIST', 'CCDI', 'CCG', 'CDDP_EAGLE',
       'CTSP', 'EXCEPTIONAL_RESPONDERS', 'FM', 'HCMI', 'MMRF', 'NCICCR',
       'OHSU', 'ORGANOID', 'RC', 'REBC', 'TRIO', 'VAREPOP', 'WCDT'],
      dtype='<U22')

### Primary sites given a program

In [6]:
gdc.url_gdc_project

'https://api.gdc.cancer.gov/projects'

In [7]:
force=False
verbose=False

prog_id = 'TCGA'

df_all_cases, df_all_samples, df_all_mutations = \
    gdc.loop_program_psi_samples(prog_id=prog_id, force=force, verbose=verbose)

print("\n----------- end ------------\n")
print(len(df_all_samples))



----------- end ------------

245657


In [8]:
df_all_mutations.columns

Index(['pid', 'barcode', 'barcode_sample', 'symbol', 'refseq_mrna_id',
       'entrez_gene_id', 'protein_mut', 'mutation_type', 'variant_type', 'chr',
       'n_mutations'],
      dtype='object')

In [ ]:
gdc.prog_id

In [ ]:
df_cases = gdc.df_cases

'pid' in df_cases.columns

In [ ]:
print(len(df_all_cases))
df_all_cases.tail(3)

In [ ]:
df_all_cases.columns

In [ ]:
cols = ['case_id', 'psi_id', 'primary_site', 'disease_type',  'diagnoses', 
       'subtype_global', 'stage_ajcc', 'primary_diagnosis', 'tumor_grade',
        'tumor_stage', 'stage', 'tumor_class', 'histology',
       'subtype_tissue'] # 'stage_clin', 'figo_stage',

print(len(df_all_cases))
df_all_cases[cols].tail(3)

In [ ]:
barcode_samples = np.unique(df_all_samples.barcode_sample)
len(barcode_samples)

In [ ]:
df_all_samples.columns

In [ ]:
sample_types = np.unique(df_all_samples.sample_type)
sample_types

In [ ]:
df_normal = df_all_samples[df_all_samples.sample_type.str.contains('Normal', case=False, na=False)]
len(df_normal)

In [ ]:
symbols = np.unique(df_all_mutations.symbol)
len(symbols)

In [ ]:
primary_sites = list(np.unique(df_all_cases.primary_site_norm))
primary_sites.sort()
print(len(primary_sites))

print(", ".join(primary_sites))

In [ ]:
df_all_mutations.columns

In [ ]:
stri = f"Interfacing GDC {prog_id} data, one gathered:"
print(stri)
stri = f"\t- {len(primary_sites)} primary sites."
print(stri)
stri = f"\t- {len(df_all_cases)} cases."
print(stri)
stri = f"\t- {len(df_all_samples)} samples."
print(stri)
stri = f"\t- {len(df_all_mutations)} annotated mutations."
print(stri)
stri = f"\t- {len(symbols)} different genes."
print(stri)


In [ ]:
primary_sites = np.array(primary_sites)
primary_sites

In [ ]:

term = 'breast'
term = 'esophagus'

primary_sites[[True if term in x.lower() else False for x in primary_sites]]

In [ ]:
primary_site = primary_sites[0]

df_cases, df_all_samples, df_all_mut, barcode_list = gdc.get_filtered_tables(primary_site=primary_site, verbose=verbose)

In [ ]:
df_all_cases.head()

In [ ]:
disease_type = 'Adenomas and Adenocarcinomas'

df2 = df_all_cases[ (df_all_cases.primary_site_norm.str.contains(term, case=False)) &
                   (df_all_cases.disease_type_norm.str.contains(disease_type, case=False)) ]
df2

In [ ]:
case_id_list = np.unique(df2.case_id)
len(case_id_list)

In [ ]:
sample_type = 'tumor'
df3 = df_all_samples[ (df_all_samples.case_id.isin(case_id_list)) & (df_all_samples.sample_type.str.contains(sample_type, case=False)) ]
print(len(df3))
df3.head(3)

In [ ]:
sample_type_list = np.unique(df3.sample_type)
sample_type_list

In [ ]:
barcode_list = list(np.unique(df3.barcode_sample))

barcode_list = ["-".join(x.split('-')[:-1]) for x in barcode_list]
len(barcode_list)

In [ ]:
np.array(barcode_list)

### Mutations

In [ ]:
df_all_mutations.columns

In [ ]:
df_all_mutations.barcode.iloc[:4]

In [ ]:
df4 = df_all_mutations[df_all_mutations.barcode.isin(barcode_list)]
len(df4), len(df_all_mutations)

In [ ]:
df4.head(3)

In [ ]:
df4["value"] = True

dfpiv = df4.pivot_table(
            index="barcode",
            columns="symbol",
            values="value",
            aggfunc="max",      # if duplicates exist
            fill_value=False )

# ensure boolean dtype
dfpiv = dfpiv.astype(bool)
print(dfpiv.shape)

Nmin_barcodes=3
Nmin_genes=5

gene_filter = dfpiv.sum(axis=0) >= Nmin_barcodes
dfpiv = dfpiv.loc[:, gene_filter]

barcode_filter = dfpiv.sum(axis=1) >= Nmin_genes
dfpiv = dfpiv.loc[barcode_filter]

print(dfpiv.shape)

dfpiv.iloc[:10,:18]

In [ ]:
df_sparse = dfpiv.astype(pd.SparseDtype("bool", fill_value=False))
df_sparse.iloc[:10,:18]

In [ ]:
print("Genes kept:", dfpiv.shape[1])
print("Samples kept:", dfpiv.shape[0])

print("\nTop mutated genes:")
print(dfpiv.sum(axis=0).sort_values(ascending=False).head(10))

print("\nMutations per sample:")
print(dfpiv.sum(axis=1).describe())

In [ ]:
import matplotlib.pyplot as plt

# number of mutated barcodes per gene
gene_counts = dfpiv.sum(axis=0).sort_values(ascending=False)

top_n = 30
top_genes = gene_counts.head(top_n)

plt.figure(figsize=(12, 6))
top_genes.plot(kind="bar")
plt.ylabel("Number of mutated barcodes")
plt.xlabel("Gene symbol")
plt.title(f"Top {top_n} mutated genes")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
gene_freq = (dfpiv.sum(axis=0) / dfpiv.shape[0]).sort_values(ascending=False)
top_genes = gene_freq.head(top_n)

plt.figure(figsize=(12, 6))
top_genes.plot(kind="bar")
plt.ylabel("Fraction of barcodes mutated")
plt.xlabel("Gene symbol")
plt.title(f"Top {top_n} mutated genes")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()

### UMAP cluster

#### Jaccard distance

Jaccard distance is a measure of dissimilarity between two sets, derived directly from Jaccard similarity. While Jaccard similarity measures how much two sets overlap, Jaccard distance measures how different they are. It is defined as one minus the Jaccard similarity.


$J(A,B) = \frac{|A \cap B|}{|A \cup B|}$

In [ ]:
from sklearn.metrics import pairwise_distances

X = dfpiv.values.astype(int)   # ensure 0/1
D = pairwise_distances(X, metric="jaccard")

### Hierarchical clustering

In [ ]:
from scipy.cluster.hierarchy import linkage

Z = linkage(D, method="average")

In [ ]:
import seaborn as sns

# reorder samples based on clustering
from scipy.cluster.hierarchy import leaves_list
order = leaves_list(Z)

df_ordered = dfpiv.iloc[order]

plt.figure(figsize=(14, 8))
sns.heatmap(df_ordered.astype(int), cmap="viridis", cbar=False)
plt.title("Mutation Heatmap (Clustered Samples)")
plt.xlabel("Genes")
plt.ylabel("Barcodes")
plt.show()

In [ ]:
sns.clustermap(
    dfpiv.astype(int),
    metric="jaccard",
    method="average",
    figsize=(14, 10),
    cmap="viridis",
    cbar=False
)

### UMAP

In [ ]:
import umap

reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    metric="jaccard",
    random_state=42
)

X = dfpiv.values.astype(int)

embedding = reducer.fit_transform(X)

In [ ]:
from sklearn.cluster import KMeans

k = 8  # tune this
labels = KMeans(n_clusters=k, random_state=42).fit_predict(embedding)

In [ ]:
colors = ['red', 'green', 'blue', 'orange', 'pink', 'purple', 'black', 'cyan']

plt.figure(figsize=(8, 6))
plt.scatter(
    embedding[:, 0],
    embedding[:, 1],
    c=[colors[label] for label in labels],
    s=20
)
plt.title("UMAP of Mutation Profiles")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.show()

In [ ]:
df_umap = pd.DataFrame(
    embedding,
    index=dfpiv.index,
    columns=["UMAP1", "UMAP2"]
)